<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part D: Deep Learning Approaches</h2>
<h2>Notebook D02: Recurrent Neural Networks</h2>
</div>

Notebook D01 ended on two admissions. The MLP had no notion that its inputs were ordered, and the dataset
was far too small for a neural network to show what it can do.

This notebook fixes both. The model reads the **sequence itself** rather than a table built from it, and we
move to a series with 35,000 observations instead of 800. Under those conditions deep learning stops being
the interesting-but-losing option it was in D01.

> This notebook needs PyTorch: `uv sync --group dl`.

---

**Contents**

1. [Imports and a Bigger Dataset](#1.-Imports-and-a-Bigger-Dataset)
2. [From a Table to Sequences](#2.-From-a-Table-to-Sequences)
3. [The Recurrent Idea](#3.-The-Recurrent-Idea)
4. [LSTM and GRU](#4.-LSTM-and-GRU)
5. [Comparing the Three](#5.-Comparing-the-Three)
6. [How Far Ahead Can It See?](#6.-How-Far-Ahead-Can-It-See?)
7. [Stacking Layers](#7.-Stacking-Layers)
8. [When to Use a Recurrent Model](#8.-When-to-Use-a-Recurrent-Model)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports-and-a-Bigger-Dataset">1. Imports and a Bigger Dataset</h3>
</div>

In [ ]:
import importlib.util
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error

import nb_config

sns.set_theme(style="whitegrid")

TORCH_AVAILABLE = importlib.util.find_spec("torch") is not None

if TORCH_AVAILABLE:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    # Small recurrent models on CPU are dominated by thread contention rather
    # than arithmetic, and run several times faster on a single thread.
    torch.set_num_threads(1)
    print(f"PyTorch {torch.__version__}, using {torch.get_num_threads()} thread")
else:
    print("PyTorch is not installed. Run 'uv sync --group dl' to follow this notebook.")

We switch to the Austrian hourly electricity load from Notebook
[B03](./B03_Advanced_statistical_models.ipynb): four years of hourly readings, with a daily cycle, a weekly
cycle, and a yearly one.

The change of dataset is not incidental. A recurrent network has to learn what a day and a week look like
purely from examples, which takes far more of them than the 734 rows D01 had to work with. It also gives
this notebook a demanding benchmark: in B03, a naive "repeat yesterday" forecast beat dynamic harmonic
regression at this exact horizon, at every origin tested.

In [ ]:
ops = pd.read_parquet(nb_config.OPS_15M_PATH)

load = (
    ops[(ops["country"] == "AT") & (ops["measure"] == "actual_entsoe_transparency")]["value"]
    .tz_convert(None)
    .resample("h").mean()
    .dropna()
    .asfreq("h")
    .loc["2016-01-01":"2019-12-31"]
)

print(f"{len(load):,} hourly observations, "
      f"{load.index.min().date()} to {load.index.max().date()}")
print(f"Load ranges from {load.min():.0f} to {load.max():.0f} MW")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-From-a-Table-to-Sequences">2. From a Table to Sequences</h3>
</div>

Part C turned the series into a table, one row per prediction, with lag columns chosen by hand. A
recurrent network wants the raw sequence instead, cut into overlapping **windows**:

- an input of the last `LOOKBACK` observations, in order;
- a target of the next `HORIZON` observations.

We use a lookback of 168 hours, one full week, and a horizon of 24 hours, the day-ahead forecast that
matters commercially. Sliding that window along the series by one hour at a time produces tens of
thousands of training examples from a single series.

Two points of discipline carry over unchanged from Part C. The split is **by time**, and the scaling
statistics come from the **training period only**.

In [ ]:
LOOKBACK = 168     # one week of history
HORIZON = 24       # forecast one day ahead

values = load.values.astype(np.float32)
n_observations = len(values)

TEST_HOURS = VALIDATION_HOURS = 24 * 90
train_end = n_observations - TEST_HOURS - VALIDATION_HOURS

# Scale using the training period only
mean, std = values[:train_end].mean(), values[:train_end].std()
scaled = (values - mean) / std


def make_windows(scaled, start, stop):
    """Every window whose target begins in [start, stop)."""
    positions = range(start, stop)
    inputs = np.stack([scaled[t - LOOKBACK:t] for t in positions])
    targets = np.stack([scaled[t:t + HORIZON] for t in positions])
    return torch.tensor(inputs)[:, :, None], torch.tensor(targets)


if TORCH_AVAILABLE:
    X_train, y_train = make_windows(scaled, LOOKBACK, train_end - HORIZON)
    X_validation, y_validation = make_windows(
        scaled, train_end, train_end + VALIDATION_HOURS - HORIZON
    )
    X_test, y_test = make_windows(
        scaled, train_end + VALIDATION_HOURS, n_observations - HORIZON
    )

    print(f"train      {tuple(X_train.shape)}  ->  {tuple(y_train.shape)}")
    print(f"validation {tuple(X_validation.shape)}  ->  {tuple(y_validation.shape)}")
    print(f"test       {tuple(X_test.shape)}  ->  {tuple(y_test.shape)}")

The shape `(30552, 168, 1)` is the convention every sequence model in PyTorch expects: **batch, time,
features**. The trailing 1 is the number of variables per timestep, and it is where additional series or
calendar channels would go in a multivariate model.

Note what is *not* in there. No lag columns, no rolling means, no day-of-week. The network gets the raw
sequence and has to work out for itself that 24 steps ago and 168 steps ago are special.

In [ ]:
def to_original_units(scaled_values):
    return np.asarray(scaled_values) * std + mean


def score(predictions, targets):
    """MAE in MW, averaged over every horizon step."""
    return mean_absolute_error(
        to_original_units(targets).ravel(), to_original_units(predictions).ravel()
    )


if TORCH_AVAILABLE:
    # The benchmark: tomorrow will look like today
    naive_positions = range(train_end + VALIDATION_HOURS, n_observations - HORIZON)
    naive_forecast = np.stack([values[t - 24:t - 24 + HORIZON] for t in naive_positions])

    NAIVE_MAE = mean_absolute_error(
        to_original_units(y_test.numpy()).ravel(), naive_forecast.ravel()
    )
    print(f"Naive baseline (repeat yesterday): {NAIVE_MAE:.1f} MW")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-The-Recurrent-Idea">3. The Recurrent Idea</h3>
</div>

A **recurrent** network reads the sequence one step at a time, carrying a **hidden state** from each step
to the next. At step $t$ it combines the new observation with everything it has accumulated so far:

$$h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$$

The same weights are used at every step. That is the crucial difference from the MLP: an MLP learned a
separate relationship for `lag_1` and `lag_7`, while a recurrent network learns **one update rule** and
applies it 168 times. Order is now intrinsic rather than encoded in column names.

The architecture here is the **RNN → fully connected** pattern: run the recurrence over the whole window,
take the final hidden state as a summary of the week, and map it to 24 outputs with one linear layer. One
forward pass produces the entire day, which makes this a **sequence-to-sequence** model in the simplest
sense.

Plain RNNs have a well-known weakness. Gradients are multiplied by the same matrix at every step on the way
back, so over 168 steps they either vanish or explode, and the network struggles to connect the target to
anything far in the past. That limitation is the reason the next section exists.

In [ ]:
class RecurrentForecaster(nn.Module):
    """Run a recurrent layer over the window, then map the last state to a forecast."""

    def __init__(self, kind="LSTM", hidden_size=64, num_layers=1, horizon=HORIZON):
        super().__init__()
        self.recurrent = getattr(nn, kind)(
            input_size=1, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
        )
        self.head = nn.Linear(hidden_size, horizon)

    def forward(self, x):
        output, _ = self.recurrent(x)      # (batch, time, hidden)
        return self.head(output[:, -1])    # the final step summarises the window


def train_recurrent(kind, num_layers=1, hidden_size=64, epochs=6,
                    batch_size=256, learning_rate=1e-3, seed=0):
    """Train one model, keeping the weights that score best on validation."""
    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)

    model = RecurrentForecaster(kind, hidden_size, num_layers)
    optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.MSELoss()

    loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=batch_size, shuffle=True, generator=generator,
    )

    started = time.time()
    best = {"validation_mae": np.inf, "epoch": 0, "weights": None}

    for epoch in range(epochs):
        model.train()
        for batch_X, batch_y in loader:
            optimiser.zero_grad()
            loss_function(model(batch_X), batch_y).backward()
            optimiser.step()

        model.eval()
        with torch.no_grad():
            validation_mae = score(model(X_validation).numpy(), y_validation.numpy())

        if validation_mae < best["validation_mae"]:
            best = {
                "validation_mae": validation_mae, "epoch": epoch,
                "weights": {k: v.clone() for k, v in model.state_dict().items()},
            }

    model.load_state_dict(best["weights"])
    model.eval()

    with torch.no_grad():
        test_predictions = model(X_test).numpy()

    return {
        "model": model,
        "predictions": test_predictions,
        "test_mae": score(test_predictions, y_test.numpy()),
        "validation_mae": best["validation_mae"],
        "best_epoch": best["epoch"],
        "parameters": sum(p.numel() for p in model.parameters()),
        "seconds": time.time() - started,
    }

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-LSTM-and-GRU">4. LSTM and GRU</h3>
</div>

Two gated architectures were designed to fix the vanishing gradient, and both are drop-in replacements
for `nn.RNN`.

**LSTM** (Long Short-Term Memory) adds a **cell state** that runs alongside the hidden state and is
modified only by addition, which lets gradients flow back over long spans without being repeatedly
multiplied. Three gates control it:

- the **forget gate** decides what to drop from the cell state;
- the **input gate** decides what new information to write;
- the **output gate** decides what to expose as the hidden state.

**GRU** (Gated Recurrent Unit) does the same job with two gates and no separate cell state:

- the **reset gate** controls how much past state to ignore when proposing an update;
- the **update gate** interpolates between keeping the old state and taking the new one.

A GRU has roughly three quarters of an LSTM's parameters and trains faster. Which wins is an empirical
question, and this dataset answers it in the next cell.

> **The next cell trains three models and takes a few minutes.**

In [ ]:
if TORCH_AVAILABLE:
    results = {}
    for kind in ("RNN", "GRU", "LSTM"):
        results[kind] = train_recurrent(kind)
        outcome = results[kind]
        print(f"{kind:<5} test MAE {outcome['test_mae']:7.1f} MW   "
              f"validation {outcome['validation_mae']:7.1f}   "
              f"{outcome['parameters']:,} parameters   "
              f"{outcome['seconds']:.0f}s")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-Comparing-the-Three">5. Comparing the Three</h3>
</div>

Three architectures, the same data, the same training budget.

In [ ]:
if TORCH_AVAILABLE:
    comparison = pd.DataFrame([
        {"Model": kind, "Test MAE": outcome["test_mae"],
         "Parameters": outcome["parameters"], "Seconds": outcome["seconds"]}
        for kind, outcome in results.items()
    ] + [{"Model": "Naive (repeat yesterday)", "Test MAE": NAIVE_MAE,
          "Parameters": 0, "Seconds": 0.0}]).sort_values("Test MAE").reset_index(drop=True)

    display_table = comparison.copy()
    display_table["vs naive"] = display_table["Test MAE"] / NAIVE_MAE

display_table.round(2)

In [ ]:
if TORCH_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5),
                             gridspec_kw={"width_ratios": [1, 1.5]})

    ordered = comparison.iloc[::-1]
    colours = ["crimson" if name.startswith("Naive") else "steelblue"
               for name in ordered["Model"]]
    axes[0].barh(ordered["Model"], ordered["Test MAE"], color=colours)
    axes[0].axvline(NAIVE_MAE, color="crimson", linestyle="--", linewidth=1.0)
    axes[0].set_title("Day-ahead MAE", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("MW")

    # One example day: the forecast from each model against what happened
    example = 24 * 30
    hours = np.arange(HORIZON)
    axes[1].plot(hours, to_original_units(y_test[example].numpy()),
                 color="black", linewidth=2.0, marker="o", markersize=4, label="Actual")
    for kind, colour in [("RNN", "darkorange"), ("GRU", "seagreen"), ("LSTM", "steelblue")]:
        axes[1].plot(hours, to_original_units(results[kind]["predictions"][example]),
                     color=colour, linewidth=1.4, linestyle="--", label=kind)
    axes[1].plot(hours, naive_forecast[example], color="crimson",
                 linewidth=1.0, linestyle=":", label="Naive")
    axes[1].set_title("One day, forecast 24 hours ahead", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Hours ahead")
    axes[1].set_ylabel("Load (MW)")
    axes[1].legend(fontsize=9)

    for ax in axes:
        ax.grid(linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

**All three recurrent models beat the naive baseline**, and the ordering is exactly what the theory
predicts: plain RNN worst, GRU better, LSTM best.

That ordering is the vanishing gradient made visible. Over a 168-step window the plain RNN cannot reliably
connect the forecast to what happened a week earlier, because the gradient signal has been multiplied by
the same matrix 168 times before it gets there. The gated architectures have a path along which gradients
travel by addition rather than multiplication, and that is the entire reason they exist.

The result is also worth putting next to Part B. In Notebook
[B03](./B03_Advanced_statistical_models.ipynb), this same naive forecast beat dynamic harmonic regression
at this same horizon, at every origin tested, because the Fourier terms described an *average* day while
the naive forecast carried today's level. The LSTM beats it by reading the actual week that just happened.
That is what the architecture buys.

Note the parameter counts. The LSTM does this with fewer than twenty thousand parameters, a fraction of the
MLP's in D01, because the recurrence shares one set of weights across all 168 steps instead of learning a
separate coefficient per lag.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-How-Far-Ahead-Can-It-See?">6. How Far Ahead Can It See?</h3>
</div>

The scores so far average over all 24 hours of the forecast. Those hours are not equally difficult, and
splitting them apart shows how fast the problem gets harder.

In [ ]:
if TORCH_AVAILABLE:
    actual_test = to_original_units(y_test.numpy())

    per_horizon = pd.DataFrame({
        kind: [mean_absolute_error(actual_test[:, h],
                                   to_original_units(outcome["predictions"])[:, h])
               for h in range(HORIZON)]
        for kind, outcome in results.items()
    })
    per_horizon["Naive"] = [
        mean_absolute_error(actual_test[:, h], naive_forecast[:, h]) for h in range(HORIZON)
    ]
    per_horizon.index = np.arange(1, HORIZON + 1)
    per_horizon.index.name = "hours ahead"

per_horizon.loc[[1, 6, 12, 18, 24]].round(0)

In [ ]:
if TORCH_AVAILABLE:
    fig, ax = plt.subplots(figsize=(12, 4.5))

    for column, colour in [("RNN", "darkorange"), ("GRU", "seagreen"),
                           ("LSTM", "steelblue"), ("Naive", "crimson")]:
        ax.plot(per_horizon.index, per_horizon[column], marker="o", markersize=4,
                linewidth=1.5, color=colour, label=column)

    ax.set_title("Error grows with the forecast horizon", fontsize=13, fontweight="bold")
    ax.set_xlabel("Hours ahead")
    ax.set_ylabel("MAE (MW)")
    ax.legend()
    ax.grid(linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

Two curves with completely different shapes, and the contrast is the most informative plot in this
notebook.

**The LSTM's error roughly doubles** between one hour ahead and twenty-four, from about 215 MW to 445 MW.
Its advantage comes from the recent past, and the further ahead it forecasts, the less that past tells it.

**The naive forecast is flat**, at around 563 MW at every horizon. That is not a bug: "the same hour
yesterday" is exactly as good a guess for hour 24 as for hour 1, so its error cannot grow with the
horizon.

The practical consequence is that the LSTM's advantage is not a single number. At one hour ahead it more
than halves the error; by twenty-four hours ahead the margin has narrowed to about 20%, and extrapolating
the two lines suggests it would close entirely not far beyond. **Quote the horizon whenever you quote an
error**, and be suspicious of any comparison that does not.

It is also worth contrasting with Notebook [B04](./B04_Probabilistic_forecasting.ipynb), where prediction
intervals on monthly temperature barely widened with the horizon. Temperature is pinned down by its
seasonal structure at any distance; electricity load is driven by weather and activity that become
genuinely less knowable the further out you look.

**Exercise.** Retrain the LSTM with `HORIZON = 1` and compare its one-step error against the first column of the table above. Does a model trained only to predict the next hour beat the 24-hour model at that one task, and what would that imply about training separate models per horizon?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-Stacking-Layers">7. Stacking Layers</h3>
</div>

The other architecture on the slides is the **stacked** RNN: feed the output sequence of one recurrent
layer into another. The first layer produces a representation at every timestep, and the second reads that
as its input sequence, which in principle lets the model build patterns at two levels of abstraction.

In PyTorch this is one argument. It is also a good opportunity to check a claim rather than repeat it.

> **This cell trains another model and takes a couple of minutes.**

In [ ]:
if TORCH_AVAILABLE:
    stacked = train_recurrent("LSTM", num_layers=2)
    single = results["LSTM"]

    depth = pd.DataFrame([
        {"Model": "LSTM, 1 layer", "Test MAE": single["test_mae"],
         "Parameters": single["parameters"], "Seconds": single["seconds"]},
        {"Model": "LSTM, 2 layers", "Test MAE": stacked["test_mae"],
         "Parameters": stacked["parameters"], "Seconds": stacked["seconds"]},
    ])

depth.round(1)

Here the second layer does pay: roughly a 5% reduction in error. It is the first time in Part D that
adding capacity has helped, and the reason is the same one that explained every previous failure to help.
This series genuinely has structure at more than one scale, a daily shape nested inside a weekly one, and
30,000 training sequences to pin down the extra parameters with.

The price is worth stating plainly, because it is easy to skip past. The second layer nearly triples the
parameters and takes two and a half times as long to train, for that 5%. Whether such a trade is worth making
depends on what the error costs you, and on a smaller dataset it would more likely have bought overfitting
instead. Try depth, and let the validation score decide rather than the assumption that deeper is
better.

**Exercise.** Add the hour of day and day of week as two extra input channels, so `X` has shape `(batch, 168, 3)` instead of `(batch, 168, 1)`. The network can currently only infer the calendar from the shape of the sequence. How much does telling it directly help?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="8.-When-to-Use-a-Recurrent-Model">8. When to Use a Recurrent Model</h3>
</div>

| Architecture | Use it when | Cost |
|---|---|---|
| **RNN** | Short sequences, or as a teaching reference | Fails on long dependencies |
| **GRU** | Most cases; the sensible default | Slightly less capacity than LSTM |
| **LSTM** | Long windows, long-range dependencies | More parameters, slower |
| **Stacked** | Structure at several scales | Roughly triples cost; bought 5% here |

What this notebook established:

**Sequence models earn their keep when the data supports them.** The same family of methods that lost to a
random forest on 734 rows in D01 beats a strong naive baseline on 30,000 sequences here. The difference is
the dataset, not the cleverness.

**Gating is not a detail.** The gap between the plain RNN and the LSTM is the whole reason those
architectures were invented, and over a 168-step window it is plainly visible.

**Weight sharing is the real advantage over an MLP.** One update rule applied 168 times, in under twenty
thousand parameters, rather than a separate coefficient for every lag you thought to include.

**Error grows with horizon**, so a forecast error means nothing without the horizon attached.

The limitation to carry into the next notebook is speed. A recurrent network processes 168 steps
**sequentially**, and nothing about that can be parallelised: step 100 cannot begin until step 99 has
finished. That is why the LSTM took minutes where the gradient boosting of Part C took under a second, and
it is the constraint the next two architectures attack from different directions.

---

The next notebook keeps the sequence input and drops the recurrence, replacing it with convolutions that
look at every position at once:
[D03 - Convolutional Networks for Time Series](./D03_Convolutional_networks.ipynb).